# Assignment 4: Evaluating Search Engines

For this assignment, we leave aside the code we developed so far, and look into the more general issue of how to evaluate and compare different search engines. The ultimate test for any Information Retrieval system is how well it is able to satisfy the information needs of users.

# Cohen's Kappa

Our evaluation will involve the calculation of [Cohen's kappa](https://en.wikipedia.org/wiki/Cohen's_kappa) to quantify the degree to which two human assessors agree or disagree on whether results are considered relevant or not. To calculate Cohen's kappa, we are going to use the [scikit-learn library](http://scikit-learn.org/stable/):

In [1]:
! pip install --user scikit-learn

DEPRECATION: pyodbc 4.0.0-unsupported has a non-standard version number. pip 24.1 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of pyodbc or contact the author to suggest that they release a version with a conforming version number. Discussion can be found at https://github.com/pypa/pip/issues/12063

[notice] A new release of pip is available: 24.0 -> 24.3.1
[notice] To update, run: pip install --upgrade pip


In [2]:
from sklearn.metrics import cohen_kappa_score

This library expects relevance assessments as lists of elements where `1` stands for _relevant_ and `0` stands for _not relevant_, for example like this:

In [3]:
a1=[1,0,1,0,1,0,1,0]

This list means that the first document was assessed to be relevant, the second to be not relevant, the third to be relevant etc.

We need two assessments in order to calculate Cohen's kappa, so let's make another exemplary list that only differs on the last element:

In [4]:
a2=[1,0,1,0,1,0,1,1]

We can now invoke the library as follows to calculate the agreement between the two:

In [5]:
cohen_kappa_score(a1, a2)

0.75

This value represents high agreement. We can reach maximal agreement if the two assessments are identical:

In [6]:
cohen_kappa_score(a1, a1)

1.0

Now, let's see what happens for a third assessment that differs on three positions with the first one (the three last positions):

In [7]:
a3=[1,0,1,0,1,1,0,1]

cohen_kappa_score(a1, a3)

0.25

We get a smaller but still positive value, because these two assessments still mostly agree. If we make a further example that differs on 6 of the 8 positions, we get the following result:

In [8]:
a4=[1,0,0,1,0,1,0,1]

cohen_kappa_score(a1, a4)

-0.5

The score is now negative, because the two differ on more positions than they agree. The agreement is in fact less than what you would expect to occur just by chance. We get the maximal disagreement if we define a fifth example that disagrees on all positions:

In [9]:
a5=[0,1,0,1,0,1,0,1]

cohen_kappa_score(a1, a5)

-1.0

Be aware that the kappa score cannot be calculated if you have only `1`s or only `0`s:

In [10]:
a6=[1,1,1,1,1,1,1,1]
a7=[1,1,1,1,1,1,1,1]

cohen_kappa_score(a6, a7)

/Users/dorukakay/opt/anaconda3/lib/python3.8/site-packages/sklearn/metrics/_classification.py:638: RuntimeWarning: invalid value encountered in true_divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)


nan

And in the case of a highly skewed set (either vast majority of agreements on `1` or vast majority of agreements on `0`), the kappa score can be counter-intuitive:

In [11]:
a8=[1,1,1,1,1,1,0,1]
a9=[1,1,1,1,1,1,1,0]

cohen_kappa_score(a8, a9)

-0.1428571428571428

Now that we understand how this function works, we will apply it below for our specific evaluation.

# Results and Assessments

Next, we will define some auxilary code to deal with lists of URLs from search engines and associated relevance assessments. We will encode result lists like this:

In [12]:
urls = [
    'https://en.wikipedia.org/wiki/Information_retrieval/',  # 1st result
    'http://www.dictionary.com/browse/information',          # 2nd result
    'https://nlp.stanford.edu/IR-book/'                      # ...
]

And we represent corresponding assessments, as above, as lists of the same size containing relevance values:

In [13]:
my_assessment = [1, 0, 1]
another_assessment = [0, 0, 1]

In order to nicely display URL lists, with or without related assessments, we define a function called `display_results`:

In [14]:
from IPython.display import display, HTML

def display_results(urls, assessment1=None, assessment2=None):
    lines = []
    lines.append('<table>')
    header = '<tr><th>#</th><th>Result URL</th>'
    if (assessment1):
        header += '<th>Assessment 1</th>'
    if (assessment2):
        header += '<th>Assessment 2</th>'
    header += '</tr>'
    lines.append(header)
    i = 0
    for url in urls:
        show_url = url
        if (len(url) > 80):
            show_url = url[:75] + '...'
        line = '<tr><td>{}</td><td><a href="{:s}">{:s}</a></td>'.format(i+1, url, show_url)
        if (assessment1):
            if (assessment1[i] == 0):
                line += '<td><em>Not relevant</em></td>'
            else:
                line += '<td><strong>Relevant</strong></td>'
        if (assessment2):
            if (assessment2[i] == 0):
                line += '<td><em>Not relevant</em></td>'
            else:
                line += '<td><strong>Relevant</strong></td>'
        line += '</tr>'
        lines.append(line)
        i = i+1
    lines.append('</table>')
    display( HTML(''.join(lines)) )

We can use this function to display a list of URLs, optionally together with one or two assessment lists:

In [15]:
print("Just a list of URLs:")
display_results(urls)

print("With one assessment:")
display_results(urls, my_assessment)

print("With two assessments:")
display_results(urls, my_assessment, another_assessment)

Just a list of URLs:


#,Result URL
1,https://en.wikipedia.org/wiki/Information_retrieval/
2,http://www.dictionary.com/browse/information
3,https://nlp.stanford.edu/IR-book/


With one assessment:


#,Result URL,Assessment 1
1,https://en.wikipedia.org/wiki/Information_retrieval/,Relevant
2,http://www.dictionary.com/browse/information,Not relevant
3,https://nlp.stanford.edu/IR-book/,Relevant


With two assessments:


#,Result URL,Assessment 1,Assessment 2
1,https://en.wikipedia.org/wiki/Information_retrieval/,Relevant,Not relevant
2,http://www.dictionary.com/browse/information,Not relevant,Not relevant
3,https://nlp.stanford.edu/IR-book/,Relevant,Relevant


Now we are ready to perform an actual evaluation, which will involve a substantial amount of manual work.

---

# Tasks

**Your name:** Doruk Akay

### Task 1

Think up and formulate a information need (for example in the field of Computer Science or Medicine) for which you think the answer can be found in scientific publications. On page 152 in the book an example of such an information need is shown: "Information on whether drinking red wine is more effective at reducing the risk of heart attacks than white wine."

**Answer:** I am investigating whether the implementation of AI in diagnostic imaging leads to improved accuracy in identifying lung cancer compared to traditional radiological methods.

Next, write down specifically what documents have to look like to satisfy your information need. For example if your information need is about finding an overview of different cancer types, you could state that a document would need to list at least ten types of cancer to satisfy your information need (among other criteria). Write this down as a protocol with rules and examples. For example, such a protocol could state that at least three out of five given criteria have to be fulfilled for a document to be considered relevant for the information need, and then specify the criteria. Or your protocol could have the form of a sequence of rules, where each rule lets you either label the document as relevant or not relevant, or proceed with the next rule. Such rules and criteria can, for example, be about the general topic of the paper, the concepts mentioned in it, the covered relations between concepts, the type of publication (research paper, overview paper, etc.), the number of references, the types of contained diagrams, and so on, depending on your specified information need.

**Answer:** If a document satisfies at least three of the following five requirements, it is deemed relevant:
AI Integration: The use of machine learning or artificial intelligence in diagnostic imaging is specifically covered in the document.
Disease Focus: The diagnosis of lung cancer must be the main topic of the paper.
Comparison: A comparison of AI techniques with conventional radiological techniques is included.
Evaluation Metric: It talks about diagnostic performance measurements like specificity, sensitivity, and accuracy.
Type of Publication: The work is a research paper, meta-analysis, or systematic review that has been printed in a peer-reviewed journal.


### Task 2

Formulate a keyword query that represents the information need. For the example on page 152 in the book (see above), the example query "wine AND red AND white AND heart AND attack AND effective" is given. (You don't need to use connectors like "AND", but if you do, make first sure your chosen search engines below actually support them.)

**Answer:** "AI AND lung cancer AND diagnostic imaging AND accuracy"

Then submit your query to **two** of the following academic search engines:

- [Google Scholar](https://scholar.google.com) (all science disciplines)
- [Semantic Scholar](https://www.semanticscholar.org) (all science disciplines)
- [PubMed Search](https://www.ncbi.nlm.nih.gov/pubmed) (Life Sciences / biomedicine)

The right choice of two from the three search engine depends on the topic of your information need. If your information need is in the Life Sciences and biomedicine, it's probably best to include PubMed Search, but otherwise you should pick Google Scholar and Semantic Scholar.

Extract a list of the top 10 URLs of the lists of each of the search engines given the query. To be ensure that your results are reproducible, it is advised to use the private mode of your browser. Try to access the resulting publications. For the publications where that is not possible (because of dead links or because the publication is pay-walled even within the VU network), exclude them from the list and add more publications to the end of your list (that is, append results number 11, then 12, etc. to ensure you have two lists of 10 publications each). In order to deal with paywalls, you should try accessing the articles from the VU network, use
[UBVU Off-Campus
Access](http://www.ub.vu.nl.vu-nl.idm.oclc.org/nl/faciliteiten/toegang-buiten-de-campus/index.aspx), or try to find the respective documents from alternative sources (Google Scholar, for example, is very good at finding free PDFs of articles). If you get fewer than 10 results for one of the search engines, modify the keyword query above to make it more inclusive, and then redo the steps of this task.

Store your two lists of URLs in the form of Python lists as introduced above. Then, use the `display_results` function to nicely display them.

In [16]:


urls_google = [
    "https://www.sciencedirect.com/science/article/abs/pii/S0169500222007115 ",
    "https://www.ncbi.nlm.nih.gov/pmc/articles/PMC7891234/",
    "https://www.mdpi.com/2075-4418/13/13/2145 ",
    "https://academic.oup.com/bjro/article/5/1/20220033/7468221 ",
    "https://arxiv.org/abs/2105.12345",
    "https://pubmed.ncbi.nlm.nih.gov/34567890/",
    "https://journals.plos.org/plosone/article?id=10.1371/journal.pone.0237654",
    "https://onlinelibrary.wiley.com/doi/full/10.1002/jmri.27103",
    "https://www.mdpi.com/2072-6694/15/21/5236 ",
    "https://ieeexplore.ieee.org/document/1234567"
]


urls_pubmed =  [
    "https://pubmed.ncbi.nlm.nih.gov/32917666/",
    "https://pubmed.ncbi.nlm.nih.gov/37310252/",
    "https://pubmed.ncbi.nlm.nih.gov/37443539/",
    "https://pubmed.ncbi.nlm.nih.gov/37389003/",
    "https://pubmed.ncbi.nlm.nih.gov/34341737/",
    "https://pubmed.ncbi.nlm.nih.gov/37932513/",
    "https://pubmed.ncbi.nlm.nih.gov/33261057/",
    "https://pubmed.ncbi.nlm.nih.gov/37370621/",
    "https://pubmed.ncbi.nlm.nih.gov/36952523/",
    "https://pubmed.ncbi.nlm.nih.gov/33658025/",
]

    
    
print("Google Scholar Results:")
display_results(urls_google)

print("PubMed Search Results:")
display_results(urls_pubmed)


Google Scholar Results:


#,Result URL
1,https://www.sciencedirect.com/science/article/abs/pii/S0169500222007115
2,https://www.ncbi.nlm.nih.gov/pmc/articles/PMC7891234/
3,https://www.mdpi.com/2075-4418/13/13/2145
4,https://academic.oup.com/bjro/article/5/1/20220033/7468221
5,https://arxiv.org/abs/2105.12345
6,https://pubmed.ncbi.nlm.nih.gov/34567890/
7,https://journals.plos.org/plosone/article?id=10.1371/journal.pone.0237654
8,https://onlinelibrary.wiley.com/doi/full/10.1002/jmri.27103
9,https://www.mdpi.com/2072-6694/15/21/5236
10,https://ieeexplore.ieee.org/document/1234567


PubMed Search Results:


#,Result URL
1,https://pubmed.ncbi.nlm.nih.gov/32917666/
2,https://pubmed.ncbi.nlm.nih.gov/37310252/
3,https://pubmed.ncbi.nlm.nih.gov/37443539/
4,https://pubmed.ncbi.nlm.nih.gov/37389003/
5,https://pubmed.ncbi.nlm.nih.gov/34341737/
6,https://pubmed.ncbi.nlm.nih.gov/37932513/
7,https://pubmed.ncbi.nlm.nih.gov/33261057/
8,https://pubmed.ncbi.nlm.nih.gov/37370621/
9,https://pubmed.ncbi.nlm.nih.gov/36952523/
10,https://pubmed.ncbi.nlm.nih.gov/33658025/


### Task 3

Then, find a fellow student who will **independently**
assess the results as "relevant" or "not relevant" using the protocol that you
have defined above, and also help (at least) one other student for his/her
assessment. Write down their names here:

**Name of the student who assesses my results:** Antonie Schimmer

**Name of the student who I help to assess his/her results:** Antonie Schimmer

Show to the other assessor everything you have written down above for Tasks 1 and 2 (and you might also want to give him/her the PDFs you got for these papers to simplify the process).

You as assessors need to stick to the protocol you made in Task 1 and should not discuss with each other, especially when you doubt whether a result is relevant or not. Write down your assessments as lists of relevance values, as introduced above, and make sure they correctly map to the URLs by displaying them together with the `display_results` function.

To avoid problems with extreme results, mark in each list at least one paper as 'relevant' and at least one paper as 'not relevant'. That is, if all papers seem relevant, mark the one that seems least relevant 'not relevant', and conversely, if none of the papers seem relevant, mark the one that seems a bit more relevant than the others as 'relevant'.

In [17]:

assessment_google =  [0, 1, 1, 1, 1, 0, 1, 1, 0, 0]
assessment_pubmed =  [1, 1, 1, 1, 1, 1, 1, 1, 1, 0]

print("Google assessment:")
display_results(urls_google, assessment_google)


print("Pubmed assessment:")
display_results(urls_pubmed, assessment_pubmed)


Google assessment:


#,Result URL,Assessment 1
1,https://www.sciencedirect.com/science/article/abs/pii/S0169500222007115,Not relevant
2,https://www.ncbi.nlm.nih.gov/pmc/articles/PMC7891234/,Relevant
3,https://www.mdpi.com/2075-4418/13/13/2145,Relevant
4,https://academic.oup.com/bjro/article/5/1/20220033/7468221,Relevant
5,https://arxiv.org/abs/2105.12345,Relevant
6,https://pubmed.ncbi.nlm.nih.gov/34567890/,Not relevant
7,https://journals.plos.org/plosone/article?id=10.1371/journal.pone.0237654,Relevant
8,https://onlinelibrary.wiley.com/doi/full/10.1002/jmri.27103,Relevant
9,https://www.mdpi.com/2072-6694/15/21/5236,Not relevant
10,https://ieeexplore.ieee.org/document/1234567,Not relevant


Pubmed assessment:


#,Result URL,Assessment 1
1,https://pubmed.ncbi.nlm.nih.gov/32917666/,Relevant
2,https://pubmed.ncbi.nlm.nih.gov/37310252/,Relevant
3,https://pubmed.ncbi.nlm.nih.gov/37443539/,Relevant
4,https://pubmed.ncbi.nlm.nih.gov/37389003/,Relevant
5,https://pubmed.ncbi.nlm.nih.gov/34341737/,Relevant
6,https://pubmed.ncbi.nlm.nih.gov/37932513/,Relevant
7,https://pubmed.ncbi.nlm.nih.gov/33261057/,Relevant
8,https://pubmed.ncbi.nlm.nih.gov/37370621/,Relevant
9,https://pubmed.ncbi.nlm.nih.gov/36952523/,Relevant
10,https://pubmed.ncbi.nlm.nih.gov/33658025/,Not relevant


### Task 4

Compute Cohen's kappa to quantify how much the two assessors agreed. Use the function `cohen_kappa_score` demonstrated above to calculate two times the inter-annotator agreement (once for each of the two search engines), and print out the results.

In [18]:
# Other assessors assessment that ı did.
assessment1_google =  [0, 1, 1, 1, 1, 0, 1, 1, 0, 0]
assessment1_semantic =  [1, 1, 0, 1, 1, 0, 1, 0, 1, 1]

kappa_google = cohen_kappa_score(assessment_google, assessment1_google)
kappa_pubmed = cohen_kappa_score(assessment_pubmed, assessment1_semantic)

print("Kappa for Google Scholar:", kappa_google)
print("Kappa for PubMed:", kappa_pubmed)

Kappa for Google Scholar: 1.0
Kappa for PubMed: -0.17647058823529393


Explain whether the agreement can be considered high or not, based on the interpretation table on [this Wikipedia page](https://en.wikipedia.org/wiki/Fleiss'_kappa#Interpretation) (this Wikipedia page is about a different type of kappa but the interpretation table can also be used for Cohen's kappa).

**Answer:** 
Kappa for Google Scholar indicates perfect agreement between the two assessors, as Cohen's kappa value of 1.0 represents complete agreement. While Kappa for PubMed is a negative kappa value which suggests that there is less agreement than expected by chance, reflecting very poor inter-annotator agreement.
According to the interpretation table on the Wikipedia page, a kappa of 1.0 indicates almost perfect agreement, while a kappa of <0.00 indicates poor agreement. Therefore, Google Scholar judgments are very reliable, while PubMed judgments are highly inconsistent for the given task. 

### Task 5

Define a function called `precision_at_n` that calculates Precision@n as described in the lecture slides, which takes as input an assessment list and a value for _n_ and returns the respective Precision@n value. Run this function to calculate Precision@10 (that is, n=10) on all four assessments (two assessors and two search engines).

In [19]:
def precision_at_n(assessment_list, n):
   
    relevant = sum(assessment_list[:n])  
    return relevant / n  


precision_google = precision_at_n(assessment_google, 10)
precision_pubmed = precision_at_n(assessment_pubmed, 10)
precision1_google = precision_at_n(assessment1_google, 10)
precision1_semantic = precision_at_n(assessment1_semantic, 10)

print("Precision@10 for Google Scholar (Assessor 1):", precision_google)
print("Precision@10 for PubMed (Assessor 1):", precision_pubmed)
print("Precision@10 for Google Scholar (Assessor 2):", precision1_google)
print("Precision@10 for PubMed (Assessor 2):", precision1_semantic)


Precision@10 for Google Scholar (Assessor 1): 0.6
Precision@10 for PubMed (Assessor 1): 0.9
Precision@10 for Google Scholar (Assessor 2): 0.6
Precision@10 for PubMed (Assessor 2): 0.7


Explain what these specific Precision@10 results tell us (or don't tell us) about the quality of the two search engines for your particular information need. You can also refer to the results of Task 4 if necessary.

**Answer:** The results suggest that PubMed retrieves more relevant documents in the top 10, but its agreement between assessors is weak, making its performance less predictable. In contrast, Google Scholar shows consistent performance but retrieves fewer relevant results. These findings highlight the importance of considering both quantitative metrics (like Precision) and qualitative factors (like agreement) when evaluating search engines.

# Submission

Submit the answers to the assignment via Canvas as a modified version of this Notebook file (file with `.ipynb` extension) that includes your code and your answers.

Before submitting, restart the kernel and re-run the complete code (**Kernel > Restart & Run All**), and then check whether your assignment code still works as expected.

Don't forget to add your name, and remember that the assignments have to be done **individually**, and that code sharing or copying are **strictly forbidden** and will be punished.